# CS5489 AIGC Detection Baseline

Train a small U-Net on the official archive with a fixed held-out validation split. The validation IoU pools foreground intersection and union at 256 × 256. It is a local selection metric.

In [ ]:
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image as DisplayImage, display
from torch.utils.data import DataLoader

from prepare_data import prepare_data
from project import CachedMaskDataset, SmallUNet, run_epoch

archive_path = Path('../AIGC Detection Dataset 2026.zip')
image_size = 256
cache = Path(f'cache{image_size}')
batch_size = 8
epochs = 10
learning_rate = 1e-3
seed = 5489
validation_fraction = 0.1
checkpoint_path = Path('model.pth')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
torch.manual_seed(seed)
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')

In [ ]:
if not (cache / 'annotation_check.json').exists():
    prepare_data(archive_path, cache, image_size)

image_index = pd.read_csv(cache / 'images.csv', dtype={'ID': str})
indices = list(range(len(image_index)))
random.Random(seed).shuffle(indices)
validation_count = round(len(indices) * validation_fraction)
validation_indices, training_indices = indices[:validation_count], indices[validation_count:]

split = {
    'train': image_index.ID.iloc[training_indices].tolist(),
    'validation': image_index.ID.iloc[validation_indices].tolist(),
}
Path('split.json').write_text(json.dumps(split, indent=2), encoding='utf-8')
print(f'Training: {len(training_indices)}, validation: {len(validation_indices)}')
display(DisplayImage(filename=str(cache / 'annotation_check.png')))

In [ ]:
train_loader = DataLoader(CachedMaskDataset(cache, training_indices), batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(CachedMaskDataset(cache, validation_indices), batch_size=batch_size)
model = SmallUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
history = []
best_iou = -1.0

for epoch in range(1, epochs + 1):
    start = time.perf_counter()
    train_loss = run_epoch(model, train_loader, device, optimizer)
    val_loss, val_iou = run_epoch(model, validation_loader, device)
    seconds = time.perf_counter() - start
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                    'val_iou': val_iou, 'seconds': seconds})
    print(f'{epoch:02d}/{epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, '
          f'val IoU {val_iou:.4f}, {seconds:.1f}s', flush=True)
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({'model_state': model.state_dict(), 'image_size': image_size}, checkpoint_path)
    pd.DataFrame(history).to_csv('training_history.csv', index=False)

print('Best validation IoU:', best_iou, '| checkpoint:', checkpoint_path)

In [ ]:
results = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), layout='constrained')
axes[0].plot(results.epoch, results.train_loss, marker='o', label='Train')
axes[0].plot(results.epoch, results.val_loss, marker='o', label='Validation')
axes[0].set(xlabel='Epoch', ylabel='BCE + Dice loss')
axes[0].legend()
axes[1].plot(results.epoch, results.val_iou, marker='o')
axes[1].set(xlabel='Epoch', ylabel='Validation foreground IoU')
for ax in axes:
    ax.grid(alpha=0.3)
fig.savefig('training_curves.png', dpi=150)
display(DisplayImage(filename='training_curves.png'))
plt.close(fig)